# SBND detector calorimetry / electric-field χ² study

Recompute per-track `chi2_muon` and `chi2_proton` with **electric-field variations** from the cathode-simulation maps in `SBND_DataMap_v3.root` (see `detector_Efield.ipynb`), following the calorimetry redo logic in `makedf/makedf.py` (`updatecalo` → `chi2pid.dedx` → `chi2pid.chi2par`).

**Workflow**
1. Load a couple of `sel_2prong` MC matched `.df` files (calovar production; same layout as `wiremod.ipynb`).
2. Inspect stored χ² (nominal vs `*_new` from calo-parameter shifts).
3. Load TH3 E-field maps and build a position-dependent sampler.
4. On a small flatcaf sample, redo dE/dx + χ² with nominal vs E-field ± envelope hits.
5. Compare distributions for `trk1` / `trk2` in the loaded event tables.

In [3]:
%load_ext autoreload
%autoreload 2

import glob as glob_module
import warnings
from os import path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from scipy.interpolate import RegularGridInterpolator
from tqdm.auto import tqdm

import sys
sys.path.append("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")

from makedf import chi2pid
from makedf.makedf import make_mchdrdf, make_trkhitdf
from makedf.util import InFV, avg_chi2
from pyanalib.split_df_helpers import get_n_split

plt.style.use("presentation.mplstyle")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

DETECTOR = "SBND_Gen1"
CALO_CV = chi2pid.CALO_VARIATIONS["CV"]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# ── Matched sel_2prong MC dfs (calovar production) ───────────────────────────
# Same pattern as wiremod.ipynb: one directory, glob *_matched.df, read evt_cv + calo shifts.

_CALOVAR_DIR = (
    "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/"
    "2026_05_16_190744__sel_2prong-mc-BNB_cosmics-calovar/merged_perTPC"
)
_NOMINAL_DIR = (
    "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/"
    "2026_05_06_073032__sel_2prong-mc-BNB_cosmics/merged_perTPC"
)

calovar_matched = sorted(glob_module.glob(path.join(_CALOVAR_DIR, "*sel_2prong*_matched.df")))
nominal_matched = sorted(glob_module.glob(path.join(_NOMINAL_DIR, "*sel_2prong*_matched.df")))

# Use two chunk files for a lightweight comparison
DF_PATHS = {
    "calovar_0000": calovar_matched[0],
    "calovar_0001": calovar_matched[1] if len(calovar_matched) > 1 else calovar_matched[0],
}
if nominal_matched:
    DF_PATHS["nominal_0000"] = nominal_matched[0]

CALO_PARAMS = ("ccal", "alpha", "beta", "R")
CALO_UNIVERSES = ("cv",) + tuple(f"{p}_{s}" for p in CALO_PARAMS for s in ("p", "m"))

print("Matched files loaded:")
for label, fpath in DF_PATHS.items():
    print(f"  {label}: {path.basename(fpath)}")

Matched files loaded:
  calovar_0000: 2026_05_16_190744__sel_2prong-mc-BNB_cosmics-calovar_merged_0000_matched.df
  calovar_0001: 2026_05_16_190744__sel_2prong-mc-BNB_cosmics-calovar_merged_0001_matched.df


In [5]:
def load_evt_universe(hdf_path, universe="cv", split=0):
    """Read one evt table from a matched calovar .df (evt_<universe>_<split>)."""
    key = f"evt_{universe}_{split}"
    return pd.read_hdf(hdf_path, key=key)


def load_matched_evt_tables(hdf_path, universes=CALO_UNIVERSES, split=0):
    out = {}
    for univ in universes:
        try:
            out[univ] = load_evt_universe(hdf_path, universe=univ, split=split)
        except KeyError:
            pass
    return out


# Load cv + a few calo-shift universes from the first two calovar chunks
evt_tables = {}
for label, fpath in DF_PATHS.items():
    if not label.startswith("calovar"):
        continue
    evt_tables[label] = load_matched_evt_tables(fpath, universes=("cv", "ccal_p", "ccal_m", "alpha_p", "R_p"))

for label, tables in evt_tables.items():
    print(f"{label}: {', '.join(f'{u} ({len(df):,} evts)' for u, df in tables.items())}")

calovar_0000: cv (12,722 evts), ccal_p (12,722 evts), ccal_m (12,722 evts), alpha_p (12,722 evts), R_p (12,722 evts)
calovar_0001: cv (12,752 evts), ccal_p (12,752 evts), ccal_m (12,752 evts), alpha_p (12,752 evts), R_p (12,752 evts)


In [6]:
def _safe_avg_chi2(trk_df, chi2_name):
    try:
        return avg_chi2(trk_df, chi2_name)
    except Exception:
        return pd.Series(dtype=float)


def chi2_summary_for_tracks(evt_df, trk_labels=("trk1", "trk2")):
    """Per-event ⟨χ²⟩ averaged over planes I0/I1/I2 for each track column group."""
    rows = []
    top = evt_df.columns.get_level_values(0).unique()
    for trk in trk_labels:
        if trk not in top:
            continue
        tdf = evt_df[trk]
        rows.append(
            pd.DataFrame(
                {
                    "chi2_muon": _safe_avg_chi2(tdf, "chi2_muon"),
                    "chi2_proton": _safe_avg_chi2(tdf, "chi2_proton"),
                    "chi2_muon_new": _safe_avg_chi2(tdf, "chi2_muon_new"),
                    "chi2_proton_new": _safe_avg_chi2(tdf, "chi2_proton_new"),
                }
            )
        )
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, axis=1, keys=[t for t in trk_labels if t in top])
    out.columns.names = ["track", "quantity"]
    return out


# Quick look: stored nominal vs calovar-recomputed χ² (cv universe)
for label, tables in evt_tables.items():
    summ = chi2_summary_for_tracks(tables["cv"])
    print(f"\n{label} / cv — mean ⟨χ²_μ⟩ (nominal / new):")
    for trk in summ.columns.get_level_values(0).unique():
        print(
            f"  {trk}: mu {summ[(trk, 'chi2_muon')].mean():.2f} → {summ[(trk, 'chi2_muon_new')].mean():.2f} | "
            f"p  {summ[(trk, 'chi2_proton')].mean():.2f} → {summ[(trk, 'chi2_proton_new')].mean():.2f}"
        )


calovar_0000 / cv — mean ⟨χ²_μ⟩ (nominal / new):
  trk1: mu 12.65 → 13.24 | p  184.30 → 185.33
  trk2: mu 26.16 → 26.03 | p  87.89 → 92.29

calovar_0001 / cv — mean ⟨χ²_μ⟩ (nominal / new):
  trk1: mu 12.47 → 12.98 | p  183.99 → 185.77
  trk2: mu 26.02 → 25.94 | p  89.03 → 93.95


In [7]:
# ── Electric-field maps (detector_Efield.ipynb) ───────────────────────────────

ROOT_PATH = (
    "/exp/sbnd/app/users/jaz8600/CathodeSimulation/"
    "localProducts_larsoft_v10_06_00_02_e26_prof/sbnd_data/v01_99/"
    "SCEoffsets/SBND_DataMap_v3.root"
)
_efield_root = uproot.open(ROOT_PATH)


def th3_axis_centers(axis):
    nbins = axis.member("fNbins")
    xmin = axis.member("fXmin")
    xmax = axis.member("fXmax")
    edges = np.linspace(xmin, xmax, nbins + 1)
    return 0.5 * (edges[:-1] + edges[1:])


def load_th3(key):
    hist = _efield_root[key]
    return {
        "values": hist.values(),
        "x": th3_axis_centers(hist.member("fXaxis")),
        "y": th3_axis_centers(hist.member("fYaxis")),
        "z": th3_axis_centers(hist.member("fZaxis")),
    }


def th3_interpolator(data):
    """Trilinear interpolator; out-of-bounds → 0."""
    vals = np.asarray(data["values"], dtype=float)
    return RegularGridInterpolator(
        (data["x"], data["y"], data["z"]),
        vals,
        bounds_error=False,
        fill_value=0.0,
    )


# Fractional |ΔE|/E maps per TPC (East / West). SBND: x < 0 → E, x ≥ 0 → W.
_efield_mag_interp = {
    tpc: th3_interpolator(load_th3(f"True_ElecField_Mag_{tpc};1"))
    for tpc in ("E", "W")
}

print(
    "Loaded E-field magnitude maps:",
    {tpc: load_th3(f"True_ElecField_Mag_{tpc};1")["values"].shape for tpc in ("E", "W")},
)

Loaded E-field magnitude maps: {'E': (49, 49, 61), 'W': (49, 49, 61)}


In [8]:
def tpc_from_x(x):
    """Map hit x [cm] to East/West TPC label used in SBND_DataMap_v3.root."""
    return np.where(np.asarray(x) < 0.0, "E", "W")


def sample_efield_delta_mag(x, y, z):
    """Fractional electric-field shift |ΔE|/E at (x,y,z) [cm]."""
    pts = np.column_stack([x, y, z])
    out = np.zeros(len(pts), dtype=float)
    for tpc, interp in _efield_mag_interp.items():
        mask = tpc_from_x(pts[:, 0]) == tpc
        if mask.any():
            out[mask] = interp(pts[mask])
    return out


def scaled_efield(nominal_efield, x, y, z, sign=0):
    """
    Apply map-based envelope on hit efield [kV/cm].

    sign=+1: E' = E * (1 + |δ|)
    sign=-1: E' = E * (1 - |δ|)
    sign= 0: E' = E (nominal, from caf hit)
    """
    nom = np.asarray(nominal_efield, dtype=float)
    delta = np.abs(sample_efield_delta_mag(x, y, z))
    if sign > 0:
        return nom * (1.0 + delta)
    if sign < 0:
        return nom * (1.0 - delta)
    return nom


def select_track_hits(hitdf, entry, slc_idx, pfp_idx):
    idx = hitdf.index
    mask = (
        (idx.get_level_values("entry") == entry)
        & (idx.get_level_values("rec.slc..index") == slc_idx)
        & (idx.get_level_values("rec.slc.reco.pfp..index") == pfp_idx)
    )
    return hitdf.loc[mask]


def chi2_plane_for_hits(hitdf, entry, slc_idx, pfp_idx, plane, is_mc, efield_sign=0, calo_params=CALO_CV):
    """
    Mirror makedf.make_trkdf(updatecalo=...): redo dE/dx then χ²_μ / χ²_p per plane.
    Returns dict with chi2_muon, chi2_proton (scalar) and ndof.
    """
    sub = select_track_hits(hitdf, entry, slc_idx, pfp_idx)
    if len(sub) == 0:
        return None

    sub = sub.copy()
    sub["efield"] = scaled_efield(sub.efield, sub.x, sub.y, sub.z, sign=efield_sign)

    dedx_redo = chi2pid.dedx(
        sub,
        gain=DETECTOR,
        calibrate=DETECTOR,
        plane=plane,
        isMC=is_mc,
        new_calo_params=calo_params,
    )
    sub["dedx_redo"] = dedx_redo

    chi2_mu, ndof_mu = chi2pid.chi2par(sub, dedxname="dedx_redo", par="muon")
    chi2_p, ndof_p = chi2pid.chi2par(sub, dedxname="dedx_redo", par="proton")

    key = sub.index.droplevel(-1)[:1]
    return {
        "chi2_muon": float(chi2_mu.loc[key]),
        "chi2_proton": float(chi2_p.loc[key]),
        "ndof_muon": float(ndof_mu.loc[key]),
        "ndof_proton": float(ndof_p.loc[key]),
        "nhits": int(len(sub)),
    }


def avg_chi2_over_planes(chi2_by_plane, name):
    vals = [v[name] for v in chi2_by_plane.values() if v is not None]
    return float(np.mean(vals)) if vals else np.nan

In [9]:
def recalc_track_chi2_efield(caf_file, entry, slc_idx, pfp_idx, is_mc, calo_params=CALO_CV):
    """Average χ² over planes for nominal / E+ / E- electric-field maps."""
    results = {}
    for sign, label in ((0, "ef_cv"), (+1, "ef_p"), (-1, "ef_m")):
        by_plane = {}
        for plane in range(3):
            hitdf = make_trkhitdf(caf_file, plane)
            hitdf = hitdf[InFV(df=hitdf, det=DETECTOR)]
            by_plane[plane] = chi2_plane_for_hits(
                hitdf, entry, slc_idx, pfp_idx, plane, is_mc,
                efield_sign=sign, calo_params=calo_params,
            )
        results[label] = {
            "chi2_muon": avg_chi2_over_planes(by_plane, "chi2_muon"),
            "chi2_proton": avg_chi2_over_planes(by_plane, "chi2_proton"),
        }
    return results


# ── Flatcaf sample for hit-level redo (same MC campaign as calovar CV) ───────
FLATCAF_LIST = (
    "/exp/sbnd/app/users/munjung/misc/filelists/MC/SBND/2025Spring_v10_06_00_10/"
    "mc_MCP2025B_1e20_10_prodgenie_corsika_proton_rockbox_sbnd_SystVar_CV_caf_flat_caf_sbnd_xrootd.list"
)
with open(FLATCAF_LIST) as _fh:
    FLATCAF_PATH = _fh.readline().strip()

print("Demo flatcaf:", path.basename(FLATCAF_PATH))
_caf = uproot.open(FLATCAF_PATH)
_ismc = bool(make_mchdrdf(_caf).ismc.iloc[0])
print("isMC:", _ismc)

Demo flatcaf: reco2-2041-fd3a-8e9f-13ee.flat.caf.root
isMC: True


In [10]:
def iter_tracks_from_evt(evt_df, max_events=20, trk_labels=("trk1", "trk2")):
    """Yield (entry, slc, pfp_idx, trk_label) for tracks present in the evt table."""
    top = evt_df.columns.get_level_values(0).unique()
    for n, idx in enumerate(evt_df.index):
        if n >= max_events:
            break
        _ntuple, entry, slc = int(idx[0]), int(idx[1]), int(idx[2])
        for trk in trk_labels:
            if trk not in top:
                continue
            pfp_idx = int(evt_df[trk].loc[idx][("pfp", "tindex", "", "", "", "")])
            yield entry, slc, pfp_idx, trk


# Match evt rows to the demo flatcaf (same entry/slc/tindex) and redo χ²
evt_cv = evt_tables["calovar_0000"]["cv"]
recalc_rows = []

for entry, slc, pfp_idx, trk_label in tqdm(list(iter_tracks_from_evt(evt_cv, max_events=25)), desc="efield χ² redo"):
    try:
        chi2_ef = recalc_track_chi2_efield(_caf, entry, slc, pfp_idx, _ismc)
    except Exception:
        continue

    row_idx = (0, entry, slc)  # __ntuple=0 for first file in merged chunk 0
    trk_one = evt_cv[trk_label].loc[[row_idx]]
    recalc_rows.append(
        {
            "entry": entry,
            "slc": slc,
            "track": trk_label,
            "pfp_idx": pfp_idx,
            "chi2_muon": float(_safe_avg_chi2(trk_one, "chi2_muon").iloc[0]),
            "chi2_muon_new": float(_safe_avg_chi2(trk_one, "chi2_muon_new").iloc[0]),
            "chi2_proton": float(_safe_avg_chi2(trk_one, "chi2_proton").iloc[0]),
            "chi2_proton_new": float(_safe_avg_chi2(trk_one, "chi2_proton_new").iloc[0]),
            "chi2_muon_ef_cv": chi2_ef["ef_cv"]["chi2_muon"],
            "chi2_muon_ef_p": chi2_ef["ef_p"]["chi2_muon"],
            "chi2_muon_ef_m": chi2_ef["ef_m"]["chi2_muon"],
            "chi2_proton_ef_cv": chi2_ef["ef_cv"]["chi2_proton"],
            "chi2_proton_ef_p": chi2_ef["ef_p"]["chi2_proton"],
            "chi2_proton_ef_m": chi2_ef["ef_m"]["chi2_proton"],
        }
    )

# Fallback: if merged-chunk events are not in this flatcaf, demo on local tracks
if not recalc_rows:
    hitdf0 = make_trkhitdf(_caf, 0)
    hitdf0 = hitdf0[InFV(df=hitdf0, det=DETECTOR)]
    keys = hitdf0.index.droplevel(-1).drop_duplicates()[:12]
    for key in keys:
        entry, slc, pfp_idx = int(key[0]), int(key[1]), int(key[2])
        chi2_ef = recalc_track_chi2_efield(_caf, entry, slc, pfp_idx, _ismc)
        recalc_rows.append(
            {
                "entry": entry,
                "slc": slc,
                "track": "flatcaf",
                "pfp_idx": pfp_idx,
                "chi2_muon_ef_cv": chi2_ef["ef_cv"]["chi2_muon"],
                "chi2_muon_ef_p": chi2_ef["ef_p"]["chi2_muon"],
                "chi2_muon_ef_m": chi2_ef["ef_m"]["chi2_muon"],
                "chi2_proton_ef_cv": chi2_ef["ef_cv"]["chi2_proton"],
                "chi2_proton_ef_p": chi2_ef["ef_p"]["chi2_proton"],
                "chi2_proton_ef_m": chi2_ef["ef_m"]["chi2_proton"],
            }
        )

recalc_df = pd.DataFrame(recalc_rows)
print(f"Tracks with hit-level E-field χ² redo: {len(recalc_df)}")
recalc_df.head(10)

efield χ² redo:   0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_3676423/2847766217.py:71: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "chi2_muon": float(chi2_mu.loc[key]),
/tmp/ipykernel_3676423/2847766217.py:72: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "chi2_proton": float(chi2_p.loc[key]),
/tmp/ipykernel_3676423/2847766217.py:73: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "ndof_muon": float(ndof_mu.loc[key]),
/tmp/ipykernel_3676423/2847766217.py:74: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "ndof_proton": float(ndof_p.loc[key]),
/tmp/ipykernel_3676423/2847766217.py:71: FutureWarning: Calling float on a single element Series is de

KeyError: "None of [MultiIndex([(0, 3, 0)],\n           names=['__ntuple', 'entry', 'rec.slc..index'])] are in the [index]"

In [ ]:
# Compare calo-shift universes (stored in .df) vs E-field redo (hit-level)

if len(recalc_df) and "chi2_muon" in recalc_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, particle, cols in zip(
        axes,
        ("muon", "proton"),
        (
            ("chi2_muon", "chi2_muon_new", "chi2_muon_ef_p", "chi2_muon_ef_m"),
            ("chi2_proton", "chi2_proton_new", "chi2_proton_ef_p", "chi2_proton_ef_m"),
        ),
    ):
        for col in cols:
            if col not in recalc_df.columns:
                continue
            vals = recalc_df[col].replace([np.inf, -np.inf], np.nan).dropna()
            if len(vals):
                ax.hist(vals, bins=30, histtype="step", label=col, density=True)
        lbl = r"$\langle\chi^2_\mu\rangle$" if particle == "muon" else r"$\langle\chi^2_p\rangle$"
        ax.set_xlabel(lbl)
        ax.legend(fontsize=8)
        ax.set_title(f"{particle} — stored vs E-field map")
    fig.tight_layout()
    plt.show()

# Envelope across calo universes in the first calovar chunk (no flatcaf needed)
univ_compare = {}
for univ, df in evt_tables["calovar_0000"].items():
    summ = chi2_summary_for_tracks(df)
    if summ.empty:
        continue
    univ_compare[univ] = {
        "chi2_muon_new": summ.xs("chi2_muon_new", axis=1, level="quantity").mean().mean(),
        "chi2_proton_new": summ.xs("chi2_proton_new", axis=1, level="quantity").mean().mean(),
    }
pd.DataFrame(univ_compare).T